In [1]:
!pip install timm

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import timm
from torchvision import transforms
from concurrent.futures import ThreadPoolExecutor

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda:5


## Image Utilities

In [4]:
def load_image(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Image not found: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def resize_image(img):
    return cv2.resize(img, (512, 512))

def extract_patches(img, patch_size=32):
    """Extract 256 non-overlapping 32x32 patches from a 512x512 image."""
    patches = []
    for i in range(0, 512, patch_size):
        for j in range(0, 512, patch_size):
            patches.append(img[i:i+patch_size, j:j+patch_size])
    return patches

In [5]:
# Normalization matches the training cache pipeline in alternaria.ipynb:
# T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]) was applied to the
# full 512x512 image before patches were extracted and fed into the backbone.
patch_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def prepare_patches(patches):
    """Stack 256 patches into tensor [256, 3, 224, 224]."""
    return torch.stack([patch_transform(p) for p in patches])

## Alternaria Disease Model

Training saved only the `MILModel` head (`torch.save(model.state_dict(), "best_model.pt")`),
so backbone and MIL are loaded separately — same pattern as Anthracnose:
- **Backbone** → `pretrained=True` (same frozen ImageNet weights used during training)
- **MIL head** → loaded directly into `MILModel()` so keys match exactly

In [6]:
class MILModel(nn.Module):
    def __init__(self, d=1280):
        super().__init__()
        self.patch_cls = nn.Linear(d, 1)
        self.V         = nn.Linear(d, 512)
        self.w         = nn.Linear(512, 1)
        self.leaf_cls  = nn.Linear(d, 1)

    def forward(self, f):
        z_i = self.patch_cls(f).squeeze(-1)                      # [B, N]
        a   = torch.softmax(
            self.w(torch.tanh(self.V(f))).squeeze(-1), dim=1
        )                                                         # [B, N]  sums to 1
        F   = (a.unsqueeze(-1) * f).sum(1)                       # [B, d]
        z   = self.leaf_cls(F).squeeze(-1)                       # [B]
        return z, z_i, a

In [8]:
# ── Backbone ──────────────────────────────────────────────────────────────────
# pretrained=True loads the same ImageNet weights used during training.
# The backbone was NEVER fine-tuned; only the MIL head was trained.
backbone = timm.create_model(
    "tf_efficientnetv2_b1",
    pretrained=True,
    num_classes=0
).cpu().eval()
print("Backbone ready (pretrained=True, frozen)")

# ── MIL head ──────────────────────────────────────────────────────────────────
# Load directly into MILModel() — keys are patch_cls.weight, V.weight, etc.
disease_model_path = "./pwdrymldwfinalplss/best_model.pt"   # <- update path if needed

mil_model   = MILModel(d=1280)
state_dict  = torch.load(disease_model_path, map_location="cpu")
clean_state = {k.replace("module.", ""): v for k, v in state_dict.items()}
mil_model.load_state_dict(clean_state)   # strict=True: all 8 keys must match
mil_model   = mil_model.cpu().eval()

print("MIL head loaded from", disease_model_path)
print("Powdery Mildew disease model ready")

Backbone ready (pretrained=True, frozen)
MIL head loaded from ./pwdrymldwfinalplss/best_model.pt
Powdery Mildew disease model ready


/tmp/ipykernel_10118/1443995508.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict  = torch.load(disease_model_path, map_location="cpu")


## Alternaria Inference

Severity = **attention-weighted patch probability sum** x 100  
Matches `infer()` in `alternaria.ipynb` exactly:
```python
severity = (probs * attn).sum() * 100
```
Grade: **Mild < 10 · Moderate < 25 · Severe < 50 · Critical >= 50**

In [9]:
def run_disease(img_path):
    """
    Run Alternaria detection on a single image.

    Returns dict:
        disease          : 'Healthy' or 'Alternaria'
        present          : bool
        leaf_probability : float  (bag-level sigmoid)
        severity_percent : float  (0-100, attention-weighted)
        grade            : 'No Disease' / 'Mild' / 'Moderate' / 'Severe' / 'Critical'
    """
    # 1) Preprocess
    img          = load_image(img_path)
    img          = resize_image(img)          # 512x512
    patches      = extract_patches(img)       # 256 x (32x32) numpy arrays
    patch_tensor = prepare_patches(patches)   # [256, 3, 224, 224]

    # 2) Feature extraction (frozen pretrained backbone)
    with torch.no_grad():
        feats = backbone(patch_tensor)        # [256, 1280]
        feats = feats.unsqueeze(0)            # [1, 256, 1280]

        # 3) MIL head
        z, z_i, a = mil_model(feats)

    # 4) Probabilities
    leaf_prob = torch.sigmoid(z).item()           # scalar: P(diseased)
    probs     = torch.sigmoid(z_i).numpy()[0]     # [256]: per-patch P(diseased)
    attn      = a.numpy()[0]                       # [256]: attention weights (sum=1)

    # 5) Healthy branch
    if leaf_prob < 0.5:
        return {
            "disease":          "Healthy",
            "present":          False,
            "leaf_probability": round(leaf_prob, 4),
            "severity_percent": 0.0,
            "grade":            "No Disease",
        }

    # 6) Diseased branch: attention-weighted severity (matches alternaria.ipynb)
    severity = float((probs * attn).sum() * 100)
    severity = max(0.0, min(severity, 100.0))

    if severity < 10:
        grade = "Mild"
    elif severity < 25:
        grade = "Moderate"
    elif severity < 50:
        grade = "Severe"
    else:
        grade = "Critical"

    return {
        "disease":          "Alternaria",
        "present":          True,
        "leaf_probability": round(leaf_prob, 4),
        "severity_percent": round(severity, 2),
        "grade":            grade,
    }

## Plant Classifier Model

In [10]:
plant_model_path = "../plantsclassifier/saved_models/best_model_fold_2.pth"   # <- update if needed

plant_model = timm.create_model("convnext_small", pretrained=False, num_classes=23)

checkpoint  = torch.load(plant_model_path, map_location="cpu")
state_dict  = checkpoint["model_state_dict"]
clean_state = {k.replace("module.", ""): v for k, v in state_dict.items()}

plant_model.load_state_dict(clean_state, strict=False)
plant_model = plant_model.cpu().eval()

print("Plant classifier ready")

/tmp/ipykernel_10118/267889131.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint  = torch.load(plant_model_path, map_location="cpu")


Plant classifier ready


In [11]:
# Matches val_transform from plants.ipynb exactly:
#   Resize(256) -> CenterCrop(224) -> Normalize
plant_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

PLANT_CLASSES = sorted([
    "appleLeaf", "bananaStem", "blackgramLeaf", "brinjalLeaf", "cassavaLeaf",
    "cauliflower", "cherryLeaf", "chicpeaPlant", "cottonLeaf", "cucumberLeaf",
    "juteLeaf", "mangoFruit", "mangoLeaf", "pomegranateFruit", "potatoLeaf",
    "pumpkinLeaf", "rice", "roseLeaf", "strawberryLeaf", "sunflowerLeaf",
    "tomatoLeaf", "watermelonLeaf", "wheatPlant"
])

def run_plant(img_path):
    img = load_image(img_path)    # numpy RGB, any size
    img = plant_transform(img)    # Resize(256) -> CenterCrop(224) -> Normalize
    img = img.unsqueeze(0)        # [1, 3, 224, 224]
    with torch.no_grad():
        logits = plant_model(img)
        pred   = torch.argmax(logits, dim=1).item()
    return PLANT_CLASSES[pred]

## Parallel Pipeline

In [12]:
def pipeline(img_path):
    """
    Run plant classification + Alternaria detection in parallel.

    Returns dict:
        plant, disease, present, leaf_probability, severity_percent, grade
    """
    with ThreadPoolExecutor(max_workers=2) as executor:
        future_disease = executor.submit(run_disease, img_path)
        future_plant   = executor.submit(run_plant,   img_path)
        disease_out    = future_disease.result()
        plant_name     = future_plant.result()

    return {"plant": plant_name, **disease_out}

## Test

In [13]:
img_path = "../pwdrymldwImplementation/pwdrymldwfinalplss/dataset/PowderyMildew/apple/81352f0c4c5efa9b.jpg"   # <- update

result = pipeline(img_path)

print("=" * 42)
print(f"Plant            : {result['plant']}")
print(f"Disease          : {result['disease']}")
print(f"Present          : {result['present']}")
print(f"Leaf Probability : {result['leaf_probability']}")
print(f"Severity         : {result['severity_percent']} %")
print(f"Grade            : {result['grade']}")
print("=" * 42)

Plant            : appleLeaf
Disease          : Alternaria
Present          : True
Leaf Probability : 0.9923
Severity         : 49.13 %
Grade            : Severe


In [14]:
img_path = "../pwdrymldwImplementation/pwdrymldwfinalplss/dataset/Healthy/apple/IMG_20190726_190938.jpg"   # <- update

result = pipeline(img_path)

print("=" * 42)
print(f"Plant            : {result['plant']}")
print(f"Disease          : {result['disease']}")
print(f"Present          : {result['present']}")
print(f"Leaf Probability : {result['leaf_probability']}")
print(f"Severity         : {result['severity_percent']} %")
print(f"Grade            : {result['grade']}")
print("=" * 42)

Plant            : appleLeaf
Disease          : Healthy
Present          : False
Leaf Probability : 0.1991
Severity         : 0.0 %
Grade            : No Disease
